# RelCheck: BLIP-2 Caption Probe on 10 Images

This notebook:
- Uses BLIP-2 to caption your images.
- Saves a CSV with `image_name, caption` for manual hallucination labeling.

In [1]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [ ]:
!pip install -q transformers accelerate torch torchvision pillow pandas

## Option A: Mount Google Drive (if your images are in Drive)

If your 10 images are stored in Drive (recommended), run this cell and set `image_folder` accordingly in the next cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Option B: Manual upload (if your images are on your laptop)

If you prefer to upload the 10 images directly from your computer, run this cell instead of using Drive. Uploaded files will be stored in `/content/`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select your 10 image files when prompted

## Set the image folder path

- If you used **Drive**, set `image_folder` to something like `/content/drive/MyDrive/relcheck_images`.
- If you used **manual upload**, set `image_folder = "/content"`.

In [ ]:
import os

# TODO: Set this to the folder containing your 10 images.
# Example for Drive:
# image_folder = "/content/drive/MyDrive/relcheck_images"

# Example for manual upload:
image_folder = "/content"  # change if needed

print("Using image folder:", image_folder)
print("Files in folder:", os.listdir(image_folder))

## Load BLIP-2 captioning model

We use `Salesforce/blip2-flan-t5-xl`, which is usually fine on Colab GPU.

In [ ]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_name = "Salesforce/blip2-flan-t5-xl"

processor = Blip2Processor.from_pretrained(model_name)
model = Blip2ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to(device)

print("Model loaded.")

## Run BLIP-2 on all images and save captions to CSV

This will:
- Loop over all `.jpg`, `.jpeg`, `.png` files in `image_folder`.
- Ask: "Describe this image in detail."
- Save `image_name, caption` as `captions_blip2.csv`.

In [ ]:
from PIL import Image
import pandas as pd

results = []

valid_exts = (".jpg", ".jpeg", ".png")

for fname in sorted(os.listdir(image_folder)):
    if not fname.lower().endswith(valid_exts):
        continue

    img_path = os.path.join(image_folder, fname)
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Skipping {fname}, error reading image:", e)
        continue

    prompt = "Describe this image in detail."
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device, torch.float16)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=80)
    caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    print(fname, "->", caption)
    results.append({"image_name": fname, "caption": caption})

df = pd.DataFrame(results)
df.to_csv("captions_blip2.csv", index=False)
df.head()

## Download the CSV to your computer

You will get `captions_blip2.csv`. Open it in Excel/Sheets and add columns like `any_hallu` and `rel_hallu`.

In [ ]:
from google.colab import files
files.download("captions_blip2.csv")